# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/720-hz/flyrank-ml-internship/blob/main/work/notebooks/Week%202/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)
**Task type: Ranking / scoring** (built on a classification signal underneath).

For a fixed weekly reviewer capacity, the deliverable isn't one yes/no call per page -- it's an ordered queue: which K pages should a content strategist look at this week? That is exactly the "which ones first?" question the framing guide maps to ranking/scoring, with a priority score as the output and Precision@K as the metric.


In [1]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"
if IN_COLAB:
  if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
  os.chdir(REPO_DIR)
else:
  while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
lane = df[df["impressions_90d"] >= 100].copy()
print(f"Reviewable pool (real demand, impressions_90d >= 100): {len(lane)} pages")
for k in [20, 50]:
  print(f"K={k}/week is {k/len(lane)*100:.2f}% of that pool -- Precision@K lives or dies on a tiny slice, so ranking (not overall accuracy) is the right frame.")

Reviewable pool (real demand, impressions_90d >= 100): 22006 pages
K=20/week is 0.09% of that pool -- Precision@K lives or dies on a tiny slice, so ranking (not overall accuracy) is the right frame.
K=50/week is 0.23% of that pool -- Precision@K lives or dies on a tiny slice, so ranking (not overall accuracy) is the right frame.


## 2. Target or proxy

The honest target is a future-observed one: did this page's traffic actually decline in the 30 days after a strategist looks at it? That label doesn't exist yet in the single 90-day-trailing-window snapshot I have -- it needs the warehouse's daily panel (fact_content_daily_performance), scored on a later month, never the sealed final month.

For this notebook I'm using a placeholder proxy: trend_direction == "down" (bucketed from trend_pct, itself an observed comparison of impressions_last_30d vs impressions_prev_30d -- not a hand-defined rule). It's real, observed movement, but it's a coarser, backward-looking signal than the true forward-looking outcome above. I'm naming that gap on purpose instead of quietly treating the proxy as the finished target.

In [2]:
df_lane = lane.copy()
df_lane["target_proxy_is_declining"] = (df_lane["trend_direction"] == "down")
print(df_lane["trend_direction"].value_counts())
print(f"\nPlaceholder proxy base rate (trend_direction == 'down'): {df_lane['target_proxy_is_declining'].mean()*100:.1f}% of the lane")

trend_direction
down      13152
stable     5232
up         3374
new         212
flat         36
Name: count, dtype: int64

Placeholder proxy base rate (trend_direction == 'down'): 59.8% of the lane


## 3. Success metric

**Precision@K** (K=20 and K=50 -- one reviewer's weekly capacity). Of the top K pages my score ranks highest, what fraction are truly declining (trend_direction == "down")? A rule that just says "flag everything" already gets 59.8% right by the lane's base rate, so Precision@K only earns its keep if it beats that 59.8% floor by a real margin -- otherwise a strategist gains nothing over grabbing pages at random.

In [3]:
base_rate = df_lane["target_proxy_is_declining"].mean() * 100
print(f"Base rate to beat (share of the lane already flagged 'down' by the proxy): {base_rate:.1f}%")
print(f"K=20 -> {20/22006*100:.2f}% of the pool; K=50 -> {50/22006*100:.2f}% of the pool -- Precision@K must clear 59.8% to beat the base rate.")

Base rate to beat (share of the lane already flagged 'down' by the proxy): 59.8%
K=20 -> 0.09% of the pool; K=50 -> 0.23% of the pool -- Precision@K must clear 59.8% to beat the base rate.


## 4. The unit of analysis, as a real dataframe

**One row = one content item, as of this snapshot.** `content_id` and `client_id` are pseudonyms for grouping and joining only, never features -- 32 clients, one row per content item per client in the 90-day-trailing snapshot.

The lane slice adds a demand floor (impressions_90d >= 100, real reviewable traffic) -- shown as `lane`/`df_lane` below. Later, once I pull the warehouse's daily panel, the grain changes to content item x period (the same content_id repeats once per report_date), which is what makes a true future-observed target possible.

In [4]:
print(f"df_lane shape: {df_lane.shape}")
df_lane[["content_id", "client_id", "content_type", "impressions_90d", "avg_position", "trend_direction"]].head()

df_lane shape: (22006, 45)


,content_id,client_id,content_type,impressions_90d,avg_position,trend_direction
0,content_304f48230142,client_f369cb89fc,keyword article,3803,10.6,down
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,20.3,down
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,36.5,down
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,6.2,stable
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,44.0,down


## 5. Why ML beats a fixed rule here

Last week's notebook (w01) already tested the obvious fixed rule -- flag every page whose trailing engagement momentum went down -- against the real trend_direction label. It caught 100% of true declines (perfect recall) but was only 76% precise: roughly 1 in 4 flagged pages weren't actually declining. A single if-statement on one signal isn't precise enough for a queue a strategist can trust.

Zooming out to more candidate signals doesn't rescue a hand-written rule either -- correlating trend_pct against 8 candidate columns in this lane, every one comes out close to zero (max |r| ~ 0.05, computed below). No single column, or an obvious one-line combination of a couple, cleanly separates decliners from the rest. That's the shape ML is built for: many weak, tangled signals worth combining, not a pattern a person can write as one rule.

I'm treating this as an open empirical question, not a foregone conclusion -- ML-07 onward is where a real rule baseline and a trained model get compared head-to-head on held-out data, and that comparison is the actual evidence, not this paragraph.

In [5]:
signal_cols = ["avg_position", "impressions_90d", "days_since_last_update", "word_count", "engagement_rate", "content_age_days", "scroll_rate", "ai_traffic_pct"]
corr = df_lane[signal_cols].corrwith(df_lane["trend_pct"]).sort_values(key=lambda s: s.abs(), ascending=False)
print("Correlation of each candidate signal with trend_pct (the label's own source), sorted by |r|:")
print(corr.round(4))

Correlation of each candidate signal with trend_pct (the label's own source), sorted by |r|:
avg_position              0.0499
impressions_90d           0.0251
days_since_last_update   -0.0159
word_count               -0.0079
engagement_rate           0.0038
content_age_days         -0.0028
scroll_rate              -0.0020
ai_traffic_pct           -0.0013
dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.